# Phase 3c v5— GEE NDVI Pull (MODIS + Sentinel-2)

**Outputs (to Drive):**
- `Drive/Quants/alternative_data/raw/ndvi_modis_srw_hrw_2008_present.csv`
- `Drive/Quants/alternative_data/raw/ndvi_s2_srw_hrw_2015_present.csv`

**Regions (state-level unions, masked to USDA NASS CDL winter-wheat pixels):**
- **SRW** (Soft Red Winter): OH, IN, IL, MO, AR, KY, TN
- **HRW** (Hard Red Winter): KS, OK, NE, CO, TX

**Method (credit-conscious):**
- Server-side reduction with `ee.ImageCollection.map(reducer)` + one `getInfo()` per region per satellite.
- CDL wheat mask = code 24 (Winter Wheat) ∪ code 26 (Winter Wheat / Soybeans double-crop), refreshed per year.
- MODIS at native 16-day cadence; Sentinel-2 composited weekly (median) before reduction.
- No thumbnails, no `getDownloadURL`, no per-pixel exports.
- Anomaly transform (52-week z-score, +1-day publication lag) is done **client-side in the ablation notebook**, not here. Keeps this pull idempotent.

**References:** see `docs/phase3_bibliography.md`. Key: Gorelick et al. 2017 (GEE), Becker-Reshef et al. 2010 (MODIS→KS wheat), Johnson 2014 (CDL+MODIS for US crops), Skakun et al. 2017 (S2 winter wheat), Fischer & Gallagher 2024 (NDVI→commodity prices).


## 1. Setup — auth, init, install eemont

In [ ]:
!pip install -q eemont earthengine-api

In [ ]:
import ee
import eemont  # noqa: F401  -- registers .maskClouds() / .scale() / .index() on ee classes
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
from google.colab import drive

ee.Authenticate()
ee.Initialize(project='vip-494502')
print('Earth Engine initialized on project vip-494502')

drive.mount('/content/drive')
ALT_RAW = Path('/content/drive/MyDrive/Quants/alternative_data/raw')
ALT_RAW.mkdir(parents=True, exist_ok=True)
print(f'Output dir: {ALT_RAW}')


## 2. Region geometries

State-level union from `TIGER/2018/States`. Single FeatureCollection per region — keeps reductions to two per satellite.


In [ ]:
STATES = ee.FeatureCollection('TIGER/2018/States')

SRW_STATES = ['OH', 'IN', 'IL', 'MO', 'AR', 'KY', 'TN']
HRW_STATES = ['KS', 'OK', 'NE', 'CO', 'TX']

def region_geom(state_abbrs):
    fc = STATES.filter(ee.Filter.inList('STUSPS', state_abbrs))
    return fc.geometry().dissolve(maxError=1000)

SRW_GEOM = region_geom(SRW_STATES)
HRW_GEOM = region_geom(HRW_STATES)

# Sanity: print areas (km^2). Single getInfo each, server-side reduction.
print('SRW area km^2:', SRW_GEOM.area(maxError=1000).divide(1e6).getInfo())
print('HRW area km^2:', HRW_GEOM.area(maxError=1000).divide(1e6).getInfo())


## 3. CDL wheat mask (per-year)

CDL is annual; we build a function that returns the wheat mask for a given year (24 = Winter Wheat, 26 = Winter Wheat/Soybeans double-crop). For dates pre-2008 (CDL coverage gap in some states pre-2008) we'd fall back to the earliest available year, but our window starts 2008-01-01 so CDL covers it.


In [ ]:
import re

CDL = ee.ImageCollection('USDA/NASS/CDL')

# CDL is annual and lags ~1 year. Pull the available indices client-side once
# so we can clamp any future-year mask request server-side. Note: CDL
# system:index entries are mostly '2008', '2017', etc. but also include
# regional sub-products like '2005a' / '2006a' (Nebraska-only releases) — we
# strip those by taking only the leading 4-digit year.
_raw_idx = CDL.aggregate_array('system:index').getInfo()
cdl_years = sorted({
    int(m.group())
    for s in _raw_idx
    if (m := re.match(r'\d{4}', str(s)))
})
MIN_CDL_YEAR, MAX_CDL_YEAR = cdl_years[0], cdl_years[-1]
print(f'CDL coverage: {MIN_CDL_YEAR} .. {MAX_CDL_YEAR}  '
      f'(any image past {MAX_CDL_YEAR} will reuse the {MAX_CDL_YEAR} mask)')

def cdl_wheat_mask(year):
    """Server-side wheat mask. Year is clamped to [MIN_CDL_YEAR, MAX_CDL_YEAR]
    so MODIS/S2 images from a not-yet-published CDL year (e.g. 2026 in early
    2026) fall back to the most recent CDL release instead of raising on a
    null image."""
    y = ee.Number(year).max(MIN_CDL_YEAR).min(MAX_CDL_YEAR)
    img = ee.Image(CDL.filter(ee.Filter.calendarRange(y, y, 'year')).first())
    crop = img.select('cropland')
    return crop.eq(24).Or(crop.eq(26)).rename('wheat_mask')

# Test on 2020 (in-range) and on a future year (would have triggered the bug).
mask_2020 = cdl_wheat_mask(2020)
print('CDL mask for 2020 built. Bands:', mask_2020.bandNames().getInfo())
mask_future = cdl_wheat_mask(MAX_CDL_YEAR + 5)
print(f'CDL mask for {MAX_CDL_YEAR + 5} built (clamped). Bands:',
      mask_future.bandNames().getInfo())


## 4. Helper — server-side NDVI reduction over a region

Returns one `(date, mean_ndvi, pixel_count)` per image in the collection. The `aggregate_array` calls keep everything server-side; only one `getInfo()` materializes the full series.


In [ ]:
def reduce_ndvi_over_region(img_coll, region_geom, ndvi_band='NDVI', scale=250):
    """Map a per-image masked-mean reduction; return server-side FeatureCollection."""
    def per_image(img):
        date = img.date().format('YYYY-MM-dd')
        year = img.date().get('year')
        mask = cdl_wheat_mask(year)
        masked = img.select(ndvi_band).updateMask(mask)
        stats = masked.reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True),
            geometry=region_geom,
            scale=scale,
            maxPixels=1e10,
            bestEffort=True,
        )
        return ee.Feature(None, {
            'date': date,
            'ndvi_mean': stats.get(f'{ndvi_band}_mean'),
            'ndvi_count': stats.get(f'{ndvi_band}_count'),
        })
    return img_coll.map(per_image)


def materialize(fc):
    """Single getInfo() to pull a FeatureCollection -> pandas DataFrame."""
    feats = fc.getInfo()['features']
    rows = [f['properties'] for f in feats]
    df = pd.DataFrame(rows)
    if len(df):
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)
    return df


def materialize_chunked(img_coll, region_geom, start_date, end_date,
                        ndvi_band='NDVI', scale=250, freq='YS'):
    """Filter the collection by client-side date chunks (year-start='YS' or
    month-start='MS' typical), reduce, getInfo, concat. Splits one giant server
    compute into many small ones.

    Use freq='YS' for MODIS (16-day cadence, modest compute per year).
    Use freq='MS' for Sentinel-2 (5-day cadence + 10m native pixels — a single
    year over a multi-state region times out)."""
    chunks = pd.date_range(start_date, end_date, freq=freq)
    if len(chunks) == 0 or chunks[0] != pd.Timestamp(start_date):
        chunks = pd.DatetimeIndex([pd.Timestamp(start_date)]).append(chunks)
    chunks = chunks.append(pd.DatetimeIndex([pd.Timestamp(end_date)]))
    chunks = chunks.unique().sort_values()

    frames = []
    for i in range(len(chunks) - 1):
        a, b = chunks[i].strftime('%Y-%m-%d'), chunks[i+1].strftime('%Y-%m-%d')
        sub = img_coll.filterDate(a, b)
        try:
            fc = reduce_ndvi_over_region(sub, region_geom, ndvi_band=ndvi_band, scale=scale)
            df = materialize(fc)
            print(f'  {a}..{b}: {len(df)} rows')
            if len(df):
                frames.append(df)
        except Exception as e:
            print(f'  {a}..{b}: FAILED ({type(e).__name__}: {str(e)[:80]})')
            continue
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(
        columns=['date', 'ndvi_mean', 'ndvi_count'])
    if len(out):
        out = out.drop_duplicates(subset='date').sort_values('date').reset_index(drop=True)
    return out


def materialize_yearly(img_coll, region_geom, start_year, end_year,
                       ndvi_band='NDVI', scale=250):
    """Backwards-compat wrapper: year chunks for MODIS."""
    return materialize_chunked(
        img_coll, region_geom,
        start_date=f'{start_year}-01-01',
        end_date=f'{end_year + 1}-01-01',
        ndvi_band=ndvi_band, scale=scale, freq='YS')


## 5. MODIS pull — `MODIS/061/MOD13Q1` (250m, 16-day, 2008+)

MOD13Q1 ships NDVI scaled by 1e4. We rescale to [-1, 1] post-reduction (cheaper than scaling the image).


In [ ]:
MODIS = (ee.ImageCollection('MODIS/061/MOD13Q1')
         .filterDate('2008-01-01', datetime.now().strftime('%Y-%m-%d'))
         .select('NDVI'))
print('MODIS image count:', MODIS.size().getInfo())


In [ ]:
# Pull SRW (chunked by year — many small compute calls, each well under timeout).
print('Reducing MODIS over SRW (year-by-year)...')
CUR_YEAR = datetime.now().year
srw_modis = materialize_yearly(MODIS, SRW_GEOM,
                               start_year=2008, end_year=CUR_YEAR,
                               ndvi_band='NDVI', scale=250)
srw_modis['ndvi_mean'] = srw_modis['ndvi_mean'] / 1e4
srw_modis = srw_modis.rename(columns={'ndvi_mean': 'ndvi_srw_mean',
                                       'ndvi_count': 'ndvi_srw_count'})
print(f'SRW MODIS rows: {len(srw_modis)}')
srw_modis.head()


In [ ]:
# Pull HRW (chunked by year — TX+KS is large, single-call would time out).
print('Reducing MODIS over HRW (year-by-year)...')
hrw_modis = materialize_yearly(MODIS, HRW_GEOM,
                               start_year=2008, end_year=CUR_YEAR,
                               ndvi_band='NDVI', scale=250)
hrw_modis['ndvi_mean'] = hrw_modis['ndvi_mean'] / 1e4
hrw_modis = hrw_modis.rename(columns={'ndvi_mean': 'ndvi_hrw_mean',
                                       'ndvi_count': 'ndvi_hrw_count'})
print(f'HRW MODIS rows: {len(hrw_modis)}')
hrw_modis.head()


In [ ]:
# Merge SRW + HRW into one MODIS file.
modis = pd.merge(srw_modis, hrw_modis, on='date', how='outer').sort_values('date')
out_modis = ALT_RAW / 'ndvi_modis_srw_hrw_2008_present.csv'
modis.to_csv(out_modis, index=False)
print(f'Wrote {len(modis)} rows to {out_modis}')
print('NaN counts:'); print(modis.isna().sum())
print('\nDate range:', modis['date'].min(), '->', modis['date'].max())
modis.tail()


## 6. Sentinel-2 pull — `COPERNICUS/S2_SR_HARMONIZED` (10m, 5-day, 2015+)

S2 is much heavier than MODIS:
- 10m vs 250m → 625× more pixels per region
- 5-day vs 16-day → ~3× more images

We tame both with **weekly median composites** before reduction. Cloud masking via `eemont`'s `.maskClouds()` (uses the SCL band).

If GEE aborts due to credit exhaustion, the MODIS file is already saved — we still have track C.


In [ ]:
S2_START = '2015-06-23'  # operational date
S2_END = datetime.now().strftime('%Y-%m-%d')

# eemont's .maskClouds() / .scale() / .index() require client-side metadata
# lookup, so they CANNOT run inside a server-side map(). Apply them ONCE to
# the full collection here.
#
# Note: the previous version of this cell built server-side weekly median
# composites before reducing. Over the HRW geometry that hit the GEE
# 'User memory limit exceeded' error: each weekly composite over KS+OK+NE+CO+TX
# at 10m holds many GB of pixels. We now skip server-side compositing and
# reduce PER-IMAGE instead — GEE only needs one image's worth of pixels in
# memory at a time. We then resample to weekly mean client-side in pandas
# (much cheaper, runs on Colab CPU, no GEE memory budget consumed).
S2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterDate(S2_START, S2_END)
        .maskClouds()
        .scale()
        .index('NDVI')
        .select('NDVI'))
print('S2 collection (preprocessed) ready. Reduction will be per-image; '
      'weekly resample happens client-side after pull.')


In [ ]:
# Pull SRW S2 — per-image, MONTH-chunked (year was too big -> timeout). scale=250.
print('Reducing S2 over SRW (per-image, month-by-month)...')
S2_END_DATE = datetime.now().strftime('%Y-%m-%d')
srw_s2_raw = materialize_chunked(S2, SRW_GEOM,
                                 start_date='2015-06-23',
                                 end_date=S2_END_DATE,
                                 ndvi_band='NDVI', scale=250, freq='MS')
print(f'SRW S2 raw rows (per-image): {len(srw_s2_raw)}')

# Save the raw per-image series too — defensive, in case the weekly resample
# choice needs to be revisited later.
srw_s2_raw.to_csv(ALT_RAW / 'ndvi_s2_srw_per_image_raw.csv', index=False)

srw_s2 = (srw_s2_raw.set_index('date')
                    .resample('W-MON')
                    .agg({'ndvi_mean': 'median', 'ndvi_count': 'sum'})
                    .dropna(subset=['ndvi_mean'])
                    .reset_index())
srw_s2 = srw_s2.rename(columns={'ndvi_mean': 'ndvi_srw_mean',
                                 'ndvi_count': 'ndvi_srw_count'})
print(f'SRW S2 weekly rows: {len(srw_s2)}')
srw_s2.head()


In [ ]:
# Pull HRW S2 — same monthly-chunked strategy. HRW is biggest; if monthly still
# times out, escalate to freq='SMS' (semi-month-start = 1st & 15th) or scale=500.
print('Reducing S2 over HRW (per-image, month-by-month)...')
hrw_s2_raw = materialize_chunked(S2, HRW_GEOM,
                                 start_date='2015-06-23',
                                 end_date=S2_END_DATE,
                                 ndvi_band='NDVI', scale=250, freq='MS')
print(f'HRW S2 raw rows (per-image): {len(hrw_s2_raw)}')
hrw_s2_raw.to_csv(ALT_RAW / 'ndvi_s2_hrw_per_image_raw.csv', index=False)

hrw_s2 = (hrw_s2_raw.set_index('date')
                    .resample('W-MON')
                    .agg({'ndvi_mean': 'median', 'ndvi_count': 'sum'})
                    .dropna(subset=['ndvi_mean'])
                    .reset_index())
hrw_s2 = hrw_s2.rename(columns={'ndvi_mean': 'ndvi_hrw_mean',
                                 'ndvi_count': 'ndvi_hrw_count'})
print(f'HRW S2 weekly rows: {len(hrw_s2)}')
hrw_s2.head()


In [ ]:
# Merge S2 SRW + HRW into one file.
s2 = pd.merge(srw_s2, hrw_s2, on='date', how='outer').sort_values('date')
out_s2 = ALT_RAW / 'ndvi_s2_srw_hrw_2015_present.csv'
s2.to_csv(out_s2, index=False)
print(f'Wrote {len(s2)} rows to {out_s2}')
print('NaN counts:'); print(s2.isna().sum())
print('\nDate range:', s2['date'].min(), '->', s2['date'].max())
s2.tail()


## 7. Validation — visual sanity check

Plot the two NDVI series. SRW + HRW should both show clear seasonal cycles (peak in spring/early summer for winter wheat). Drought years (2012, 2022) should show visibly lower peaks for HRW.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(modis['date'], modis['ndvi_srw_mean'], label='SRW', alpha=0.8)
axes[0].plot(modis['date'], modis['ndvi_hrw_mean'], label='HRW', alpha=0.8)
axes[0].set_title('MODIS MOD13Q1 — wheat-masked mean NDVI')
axes[0].set_ylabel('NDVI')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(s2['date'], s2['ndvi_srw_mean'], label='SRW', alpha=0.8)
axes[1].plot(s2['date'], s2['ndvi_hrw_mean'], label='HRW', alpha=0.8)
axes[1].set_title('Sentinel-2 SR Harmonized — wheat-masked mean NDVI (weekly median)')
axes[1].set_ylabel('NDVI')
axes[1].set_xlabel('Date')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 8. References (also in `docs/phase3_bibliography.md`)

- Gorelick et al. (2017). *Google Earth Engine: Planetary-scale geospatial analysis for everyone.* RSE 202, 18–27. https://doi.org/10.1016/j.rse.2017.06.031
- Becker-Reshef et al. (2010). *A generalized regression-based model for forecasting winter wheat yields in Kansas and Ukraine using MODIS data.* RSE 114(6), 1312–1323.
- Johnson, D. M. (2014). *An assessment of pre- and within-season remotely sensed variables for forecasting corn and soybean yields in the United States.* RSE 141, 116–128.
- Skakun et al. (2017). *Early season large-area winter crop mapping and yield prediction with Sentinel-2.* RSE 195, 244–258.
- Fischer & Gallagher (2024). *Satellite-based vegetation indices and agricultural commodity returns.* J. Commodity Markets (SSRN 4281572).
- Montero, D. (2021). *eemont.* JOSS 6(62), 3168.
- USDA NASS Cropland Data Layer (CDL). GEE asset `USDA/NASS/CDL`.
